In [7]:
import pandas as pd, glob, os

# CSV ko current folder ya Downloads mein khud dhoondho
files = glob.glob("teaching_prices_messy*.csv") + \
        glob.glob(os.path.expanduser("~/Downloads/teaching_prices_messy*.csv"))
print("Found:", files)

raw = pd.read_csv(files[0], dtype=str)
df = raw.drop_duplicates().copy()

df["price_raw"] = df["price"]
df["item_raw"] = df["item"]
df["flag"] = ""

df["date"] = pd.to_datetime(df["date"])
df["price"] = pd.to_numeric(df["price"].str.replace(",", ""), errors="coerce")

df.loc[df["price_raw"].isna(), "flag"] = "blank in source"

bad = df["price"] > 1000
df.loc[bad, "flag"] = "implausible, set to missing"
df.loc[bad, "price"] = None

oil = df["item"] == "oil b"
df.loc[oil, "item"] = "Staple A"
df.loc[oil, "flag"] = "relabelled oil b to Staple A (inference)"

print(len(df), df["price"].isna().sum(), df.duplicated(["date", "item", "city"]).sum())  # 96 3 0
print(df[df["flag"] != ""][["date", "item", "city", "price", "price_raw", "flag"]])

Found: ['C:\\Users\\CBM/Downloads\\teaching_prices_messy (3).csv']
96 3 0
         date         item    city  price price_raw  \
5  2026-01-01      Pulse C  City 2    NaN       NaN   
17 2026-01-15     Staple A  City 2    NaN     1,250   
30 2026-01-22  Sweetener D  City 1    NaN      9999   
40 2026-02-05     Staple A  City 1  108.5     108.5   

                                        flag  
5                            blank in source  
17               implausible, set to missing  
30               implausible, set to missing  
40  relabelled oil b to Staple A (inference)  
